In [ ]:
%load_ext autoreload
%autoreload 2

from os.path import join as pjoin
import pandas as pd
from datetime import datetime
from bmp_config import path_data_passive as path_data
from bmp_behav_proc import *

fnf = pjoin(path_data,'df_all_multi_tsz__.pkl.zip')
print(fnf)
print( str(datetime.fromtimestamp(os.stat(fnf).st_mtime)))
df_all_multi_tsz = pd.read_pickle(fnf)
num_sessions = 1
if num_sessions == 1:
    df_all_multi_tsz = df_all_multi_tsz[df_all_multi_tsz['session_id']==1]

df = df_all_multi_tsz.query('trial_shift_size == 1 and trial_group_col_calc == "trialwe" '
                           ' and retention_factor_s == "0.924"').copy().sort_values(['subject','trials'])
dfs_ = []
for sid in df['session_id'].unique():
    print(f'Processing session {sid}')
    df_sess = df.query('session_id == @sid').copy()
    print(f'Session {sid} has {df_sess.shape[0]} rows')
    _,dfall_,ES_thr,envv,pert = addBehavCols2(df_sess);
    dfs_ += [dfall_]
dfall = pd.concat(dfs_, ignore_index=True)
s = df.groupby(['subject','session_id'], observed=True).size().min()
print(s)
assert s == 336

## No savings

In [ ]:
# Time between the end of 1st perturbation stage and beginning of 3rd perturbation stage
pairs = [(1,5),(3,7)]
for pair in pairs:
    subj2dif = {}
    for subj in dfall['subject'].unique():
        dfsubj = dfall[dfall['subject'] == subj]
        pstime = []
        for pert_stage in pair:
            dfps = dfsubj[dfsubj['pert_stage'] == pert_stage]
            dfps = dfps.sort_values('trials')
            if pert_stage in [1,3]:
                pstime += [dfps.iloc[-1]['time']]
            else:
                pstime += [dfps.iloc[0]['time']]
        subj2dif[subj] = pstime[1] - pstime[0]
            
    display(pd.Series(subj2dif).describe())

In [ ]:
ttrs2_sig_mainpairs, ttrs2_sig_otherpairs = \
checkSavingsNIH(dfall, exp_name='NIH_passive', coln_to_corr='err_sens')
ttrs2_sig_otherpairs

In [ ]:
ttrs2_sig_mainpairs, ttrs2_sig_otherpairs = \
    checkSavingsNIH(dfall, exp_name='NIH_passive', coln_to_corr='error_pscadj')
ttrs2_sig_otherpairs